## Ноутбук 03: Эксперименты, валидация и моделирование (RecSys Experiments & Modeling)

### Цели и задачи исследования:
1. **Ablation Study порогов K-core:** Экспериментальное сравнение вариантов фильтрации `(3,5)`, `(3,2)`, `(3,1)` по оффлайн-метрикам для выбора оптимального графа.
2. **Сравнение сигналов взаимодействия (Binary vs Weighted):** Сравнительный анализ моделей при использовании бинарного сигнала (`review = 1`) и взвешенного времени игры ($\log(1 + \text{playtime})$).
3. **Хронологический Train/Test Split:** Построение временного разбиения (`Leave-Last-1-Out` / `Temporal Split`) для исключения утечки данных из будущего (Data Leakage).
4. **Обучение и валидация бейзлайнов и моделей:** Оценка работы базовых и коллаборативных алгоритмов (Popularity, ItemKNN, Implicit ALS, BPR).
5. **Оценка качества и финализация:** Сравнение моделей по метрикам Recall@K, Precision@K, NDCG@K и фиксация лучшего пайплайна Candidate Generation (Retrieval).

### Анализ разреженности

In [ ]:
def print_interactions_stats(df: pl.DataFrame, title: str = "Статистика графа") -> None:
    n_users = df['author_steamid'].n_unique()
    n_items = df['appid'].n_unique()
    n_interactions = len(df)
    sparsity = 100.0 * (1.0 - (n_interactions / (n_users * n_items)))

    user_counts = df.group_by('author_steamid').len()['len']
    game_counts = df.group_by('appid').len()['len']

    print(f"=== {title} ===")
    print(f"Уникальных пользователей: {n_users:,}")
    print(f"Уникальных игр:           {n_items:,}")
    print(f"Взаимодействий (отзывов): {n_interactions:,}")
    print(f"Разреженность (Sparsity): {sparsity:.4f}%\n")

    print("--- Отзывов на пользователя ---")
    print(f"Среднее: {user_counts.mean():.2f}")
    print(f"Квантили (50%, 90%, 99%): {user_counts.quantile(0.5)}, {user_counts.quantile(0.9)}, {user_counts.quantile(0.99)}\n")

    print("--- Отзывов на игру ---")
    print(f"Среднее: {game_counts.mean():.2f}")
    print(f"Квантили (50%, 90%, 99%): {game_counts.quantile(0.5)}, {game_counts.quantile(0.9)}, {game_counts.quantile(0.99)}\n")


# Вызов для исходного датасета:
print_interactions_stats(reviews_df, title="Сырой граф")

=== Сырой граф ===
Уникальных пользователей: 701,884
Уникальных игр:           117,311
Взаимодействий (отзывов): 1,048,148
Разреженность (Sparsity): 99.9987%

--- Отзывов на пользователя ---
Среднее: 1.49
Квантили (50%, 90%, 99%): 1.0, 2.0, 8.0

--- Отзывов на игру ---
Среднее: 8.93
Квантили (50%, 90%, 99%): 2.0, 22.0, 100.0



Разреженность очень высокая, необходимо ...

In [ ]:
def filter_k_core(df: pl.DataFrame, min_user_reviews: int = 3, min_item_reviews: int = 5) -> pl.DataFrame:
    filtered_df = df
    start_len = 0
    while start_len != len(filtered_df):
        start_len = len(filtered_df)

        active_users = (filtered_df
            .group_by('author_steamid').len()
            .filter(pl.col('len') >= min_user_reviews)
        )
        active_games = (filtered_df
            .group_by('appid').len()
            .filter(pl.col('len') >= min_item_reviews)
        )
        filtered_df = (filtered_df
            .join(active_users, on='author_steamid', how='semi')
            .join(active_games, on='appid', how='semi')
        )

    return filtered_df

reviews_filtered = filter_k_core(reviews_df)

print_interactions_stats(reviews_filtered)

=== Статистика графа ===
Уникальных пользователей: 28,813
Уникальных игр:           16,437
Взаимодействий (отзывов): 188,007
Разреженность (Sparsity): 99.9603%

--- Отзывов на пользователя ---
Среднее: 6.53
Квантили (50%, 90%, 99%): 4.0, 11.0, 47.0

--- Отзывов на игру ---
Среднее: 11.44
Квантили (50%, 90%, 99%): 9.0, 20.0, 48.0



In [ ]:
reviews_filtered = filter_k_core(reviews_df, 3, 1)

print_interactions_stats(reviews_filtered)

=== Статистика графа ===
Уникальных пользователей: 47,337
Уникальных игр:           82,010
Взаимодействий (отзывов): 329,463
Разреженность (Sparsity): 99.9915%

--- Отзывов на пользователя ---
Среднее: 6.96
Квантили (50%, 90%, 99%): 4.0, 11.0, 50.0

--- Отзывов на игру ---
Среднее: 4.02
Квантили (50%, 90%, 99%): 2.0, 10.0, 29.0

